In [ ]:
# Download the repository
!git clone -q https://github.com/yasahi-hpc/JAX-PyTorch-Vlasov.git

# Make the source directory importable
import sys
sys.path.insert(0, "JAX-PyTorch-Vlasov/simulations/vlasov1d_1v/jax/src")

In [ ]:
!pip install "xarray[complete]"

In [ ]:
import os
import jax.numpy as jnp
from vlasov1D1V import (
    run_vlp1d1v
)

nbiter, diag_steps = 10, 20
device = 'tpu'
out_dir = 'data_python'
physics_mode = True
lx, vmax = 4.0*jnp.pi, 5.0
dt = 0.05
epsilon = 0.001
device_name = 'TPUv5' if device == 'tpu' else 'cpu'

for dtype in ["float32"]:
    for n in [128, 256, 512, 1024, 2048, 4096]:
        for solver in [0, 1, 2]:
            run_vlp1d1v(
                nx=n,
                nv=n,
                lx=lx,
                vmax=vmax,
                nbiter=nbiter,
                diag_steps=diag_steps,
                dt=dt,
                out_dir=out_dir,
                physics_mode=physics_mode,
                epsilon=epsilon,
                solver_type=solver,
                dtype=dtype
            )

            # Rename the resulting file to the requested format
            old_filename = f'vlp1d_1v_{dtype}.txt'
            new_filename = f'vlp1d_1v_{device_name}_{dtype}_N{n}_solver{solver}.txt'
            if os.path.exists(old_filename):
                os.rename(old_filename, new_filename)
                print(f'Renamed {old_filename} to {new_filename}')

In [ ]:
import zipfile
import os
from google.colab import files

# Define the name of the output zip file
zip_filename = 'vlp1d-1v_results.zip'

# Mapping of local directories to their desired names inside the zip
dump_mapping = {
    'jaxpr_dump': f'vlp1d_1v_{device_name}_{dtype}_jaxpr_dump',
    'hlo_dump': f'vlp1d_1v_{device_name}_{dtype}_hlo_dump'
}

# Create a zip archive
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # Add .txt files from the current directory
    txt_files = [f for f in os.listdir('.') if f.startswith('vlp1d_1v_') and f.endswith('.txt')]
    for file in txt_files:
        zipf.write(file)

    # Add files from dump folders into renamed subdirectories
    for local_folder, zip_folder in dump_mapping.items():
        if os.path.exists(local_folder):
            for root, dirs, files_in_dir in os.walk(local_folder):
                for file in files_in_dir:
                    file_path = os.path.join(root, file)
                    # Construct the internal path: new_folder_name/filename
                    archive_path = os.path.join(zip_folder, file)
                    zipf.write(file_path, arcname=archive_path)

# Download the zip file
files.download(zip_filename)